# Build Train/Test Datasets with Target and Control Patients

This notebook orchestrates building train/test datasets for final model training, ensuring **both target and control patients** are included in all splits.

## Overview

This notebook runs the complete dataset preparation pipeline:

1. **Build Final Feature Table** - Merges all feature engineering outputs (FP-Growth, BupaR, DTW, PGx, Predictive Time)
2. **Remove Target Leakage** - Removes post-event features, time-to-target features, and DTW features
3. **Prepare Train/Test Splits** - Creates temporal validation splits (2016-2018 train, 2019 test) with both target and control

## Key Features

✅ **Both Target and Control Patients** - All feature engineering includes both patient types  
✅ **Temporal Validation** - Train on 2016-2018, test on 2019 (no data leakage)  
✅ **Target Leakage Removal** - Automatic removal of post-event and time-to-target features  
✅ **Idempotent Workflow** - Skips steps if outputs already exist  
✅ **Verification** - Validates that both target and control patients are in train/test splits  

## Output Files

- **Final Features**: `6_final_model/outputs/{cohort}/{age_band}/{cohort}_{age_band}_train_final_features_no_leakage.csv`
- **Train Dataset**: `6_final_model/outputs/{cohort}/{age_band}/inputs/model_train/final_features.parquet`
- **Test Dataset**: `6_final_model/outputs/{cohort}/{age_band}/inputs/model_test/final_features.parquet`
- **S3 Location**: `s3://pgxdatalake/gold/final_model/{cohort}/{age_band}/inputs/`


## Environment Setup


In [ ]:
import os
import sys
from pathlib import Path

def resolve_python_bin() -> Path:
    env_bin = os.environ.get("COHORT_RUNNER_PYTHON")
    if env_bin:
        return Path(env_bin)
    return Path(sys.executable)

PYTHON_BIN = resolve_python_bin()
print(f"[INFO] Python binary: {PYTHON_BIN}")

def resolve_project_root() -> Path:
    if "__file__" in globals():
        root = Path(__file__).resolve().parent
        return root.parent if root.name == "6_final_model" else root
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "6_final_model" else cwd

PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Using the Orchestration Script

For a complete workflow (build features + remove leakage + create splits), use the orchestration script:

```python
# Example: Run complete workflow for a cohort/age band
import subprocess

subprocess.run(
    [str(PYTHON_BIN), "6_final_model/run_final_model.py", 
     "--cohort", "opioid_ed", 
     "--age_band", "0-12",
     "--skip-training",  # Skip model training, only build datasets
     "--skip-visualizations"],  # Skip visualizations
    cwd=PROJECT_ROOT,
)
```

**Benefits:**
- Runs all dataset preparation steps in one command
- Can be called from notebooks or command line
- Includes verification automatically

**Alternative:** Use individual cells below for more control over specific steps.


## Per-Cohort Runner Cells

Each cell below runs the complete dataset preparation pipeline for a **single cohort/age-band combination**. This makes it easy to:

- Debug failures for a specific cohort/age-band
- Modify scripts and immediately re-run just that cohort
- Verify outputs for individual cohorts

All scripts are idempotent (skip completed steps if outputs already exist).


### falls / 65–74

In [ ]:
import subprocess

COHORT, AGE_BAND = "falls", "65-74"

for step_script, label in [
    ("6_final_model/build_final_cohort_model_features.py", "Build feature table"),
    ("6_final_model/remove_target_leakage.py", "Remove target leakage"),
    ("6_final_model/prepare_train_test_s3.py", "Prepare train/test splits"),
]:
    print(f"[{label}] {COHORT} / {AGE_BAND}")
    r = subprocess.run(
        [str(PYTHON_BIN), step_script, "--cohort", COHORT, "--age_band", AGE_BAND],
        cwd=PROJECT_ROOT,
    )
    if r.returncode != 0:
        raise RuntimeError(f"{label} failed for {COHORT}/{AGE_BAND} (exit {r.returncode})")
    print(f"  OK")

print(f"Dataset preparation complete: {COHORT} / {AGE_BAND}")

### Verify Train/Test Splits Include Both Target and Control

Use this cell to verify that both target and control patients are included in the train/test splits.


In [ ]:
import pandas as pd

COHORT, AGE_BAND = "falls", "65-74"
ab_f = AGE_BAND.replace("-", "_")

for split in ("model_train", "model_test"):
    path = PROJECT_ROOT / "6_final_model" / "outputs" / COHORT / ab_f / "inputs" / split / "final_features.parquet"
    if not path.exists():
        print(f"[{split}] NOT FOUND: {path}")
        continue
    df = pd.read_parquet(path)
    counts = df["fall_injury_any"].value_counts() if "fall_injury_any" in df.columns else df.get("target", df.iloc[:, -1]).value_counts()
    print(f"[{split}] {len(df):,} rows  |  target=1: {counts.get(1, 0):,}  control=0: {counts.get(0, 0):,}")

## Run All Cohorts Sequentially

Use this section to run dataset preparation for all cohorts/age bands sequentially.


In [ ]:
import subprocess

# 65–85 group: falls and ed × 65-74 and 75-84
COHORT_BANDS = [
    ("falls", "65-74"),
    ("falls", "75-84"),
    ("ed",    "65-74"),
    ("ed",    "75-84"),
]
FAIL_FAST = True

for cohort, age_band in COHORT_BANDS:
    print(f"\n{'='*60}")
    print(f"Processing: {cohort} / {age_band}")
    for step_script, label in [
        ("6_final_model/build_final_cohort_model_features.py", "Build features"),
        ("6_final_model/remove_target_leakage.py",             "Remove leakage"),
        ("6_final_model/prepare_train_test_s3.py",             "Prepare splits"),
    ]:
        r = subprocess.run(
            [str(PYTHON_BIN), step_script, "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT,
        )
        if r.returncode != 0:
            msg = f"{label} failed for {cohort}/{age_band}"
            print(f"  FAILED: {msg}")
            if FAIL_FAST:
                raise RuntimeError(msg)
        else:
            print(f"  OK: {label}")

print("\nAll cohort dataset preparations complete.")

## Verify All Cohorts

Use this cell to verify that all cohorts have train/test splits with both target and control patients.


In [ ]:
import pandas as pd

COHORT_BANDS = [
    ("falls", "65-74"),
    ("falls", "75-84"),
    ("ed",    "65-74"),
    ("ed",    "75-84"),
]

TARGET_COL = {"falls": "fall_injury_any", "ed": "ed_event"}

print(f"{'='*70}")
print(f"{'Cohort/Band':<20} {'Split':<14} {'Rows':>8} {'target=1':>10} {'control=0':>10}")
print(f"{'='*70}")

all_ok = True
for cohort, age_band in COHORT_BANDS:
    ab_f = age_band.replace("-", "_")
    tcol = TARGET_COL[cohort]
    for split in ("model_train", "model_test"):
        path = PROJECT_ROOT / "6_final_model" / "outputs" / cohort / ab_f / "inputs" / split / "final_features.parquet"
        if not path.exists():
            print(f"  {cohort}/{age_band:<10} {split:<14} MISSING")
            all_ok = False
            continue
        df = pd.read_parquet(path)
        col = tcol if tcol in df.columns else ("target" if "target" in df.columns else None)
        if col:
            cnt = df[col].value_counts()
            n1, n0 = cnt.get(1, 0), cnt.get(0, 0)
            flag = "" if (n1 > 0 and n0 > 0) else " ⚠"
            print(f"  {cohort}/{age_band:<10} {split:<14} {len(df):>8,} {n1:>10,} {n0:>10,}{flag}")
            if flag:
                all_ok = False
        else:
            print(f"  {cohort}/{age_band:<10} {split:<14} {len(df):>8,} (target col missing) ⚠")
            all_ok = False

print(f"{'='*70}")
print("All splits verified OK." if all_ok else "Some splits have issues — see warnings above.")